# Minestone 3: Alignment-based approach

In [30]:
import pm4py
import pandas as pd
import os
import csv
import json
import ast
from collections import Counter
from pathlib import Path

from pm4py.objects.log.obj import EventLog, Trace

from pm4py.statistics.variants.log import get as variants_get
from pm4py.algo.conformance.alignments.petri_net import algorithm as aligner

from pm4py.visualization.petri_net import visualizer as pn_visualizer
from pm4py.objects.conversion.process_tree import converter as pt_converter

In [31]:
DATA_DIR = Path.cwd().parent / "data"
OUTPUT_DIR = DATA_DIR / "milestone3"

# Choose which log to use for alignment, dont process all three at once, as it will take a long time to compute
CASE = "recovered"  # choose one from ["clean", "noised", "recovered"]

if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Recompute the process model from inductive mining again
- RUN ONCE!!! - uncomment to rerun
- In the milestone 2 we have the runtime problem so we worked on the samples
- For this milestone 3 we work on the original log again

In [32]:
# log_clean = pm4py.read_xes(str(DATA_DIR / "BPI Challenge 2017.xes.gz"))
# print(f"Loaded clean log: {len(log_clean)} events, "
#         f"{log_clean['case:concept:name'].nunique()} cases")

# log_clean = pm4py.convert_to_event_log(log_clean)

In [33]:
# tree_clean = pm4py.discovery.discover_process_tree_inductive(log_clean)
# net_clean, im_clean, fm_clean = pt_converter.apply(tree_clean)

In [34]:
# gviz = pn_visualizer.apply(net_clean, im_clean, fm_clean)
# pn_visualizer.view(gviz)

In [35]:
# pm4py.write_pnml(net_clean, im_clean, fm_clean, OUTPUT_DIR / "reference_model_clean.pnml")

## Alignment-Based Conformance

In [36]:
if CASE == "clean":
    log =  pm4py.read_xes(str(DATA_DIR / "BPI Challenge 2017.xes.gz"))
elif CASE == "noised":
    log = pm4py.read_xes(str(DATA_DIR / "noised.xes.gz"))
else: # CASE == "recovered":
    log = pm4py.read_xes(str(DATA_DIR / "recovered.xes.gz"))
print(f"Loaded {CASE} log: {len(log)} events, "
        f"{log['case:concept:name'].nunique()} cases")

# just to make sure that the logs are in the right format for pm4py
log = pm4py.convert_to_event_log(log)

parsing log, completed traces :: 100%|██████████| 31509/31509 [01:01<00:00, 510.27it/s]


Loaded recovered log: 1202286 events, 31509 cases


In [37]:
net_clean, im_clean, fm_clean = pm4py.read_pnml(f"{OUTPUT_DIR}/reference_model_clean.pnml")

In [38]:
variants = variants_get.get_variants(log)
print(len(variants))

16471


### Task a: Compute optimal alignments
- For each trace variant, compute the optimal alignment with the model under the standard cost function (Lecture 11, slide 6). 
- Record, per variant (rows), a table with columns: frequency, alignment cost, move sequence (the alignment), model moves, and log moves. (This per-variant table goes in the repository.)

In [39]:
def classify_and_cost(aligned_trace):
    """
    PM4Py convention:
        - l == ">>"   -> no log move here
        - m == ">>"   -> no model move here
        - m is None   -> model move fired, but on an invisible/tau transition
    Cost function (lecture 11 slide 6):
        - synchronous move: cost 0
        - log move: cost 1
        - model move: cost 1
        - invalid synchronous move: cost infinity
    """
    cost = 0
    log_moves = []
    model_moves = []

    for l, m in aligned_trace:
        log_skip = (l == ">>")
        model_skip = (m == ">>")

        if not log_skip and not model_skip: # SYNCHRONOUS MOVE
            if l != m:
                return float("inf"), [], []
            continue

        elif log_skip and not model_skip: # MODEL MOVE
            if m is not None:  # ignore invisible/tau transitions
                cost += 1
                model_moves.append(m)

        elif model_skip and not log_skip: # LOG MOVE
            if l is not None:  # ignore invisible/tau transitions
                cost += 1
                log_moves.append(l)

    return cost, model_moves, log_moves

In [42]:
output_file = f"alignment_per_variant_{CASE}.csv"
output_path = OUTPUT_DIR / output_file

BATCH_SIZE = 1000
batch = []


# Load checkpoint
if os.path.exists(OUTPUT_DIR / output_file):
    df_existing = pd.read_csv(OUTPUT_DIR / output_file)
    done_variants = set(df_existing["variant"].astype(str))
else:
    done_variants = set()   

processed = 0
for variant, traces in variants.items():
    processed += 1
    if str(variant) in done_variants:
        print(f"Skipping {processed}/{len(variants)}: {variant}")
        continue
    
    # Only need to align one representative trace per variant, since all traces in a variant are identical
    representative = traces[0]

    alignment = aligner.apply_trace(
        representative,
        net_clean,
        im_clean,
        fm_clean
    )

    aligned = alignment["alignment"]

    # Manually classify the alignment and compute the cost of deviations instead of relying on the cost computed by pm4py
    cost, model_moves, log_moves = classify_and_cost(aligned)

    row = {
        "variant": variant,
        "frequency": len(traces),
        "cost": cost,
        "alignment": json.dumps(aligned),
        "model_moves": json.dumps(model_moves),
        "log_moves": json.dumps(log_moves)
    }

    batch.append(row)
    print(f"Processed {processed}/{len(variants)}: {variant} | cost={cost}")

    if len(batch) >= BATCH_SIZE:
        file_exists = os.path.exists(output_path)

        with open(output_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=batch[0].keys())

            if not file_exists:
                writer.writeheader()

            writer.writerows(batch)

        print(f"Saved batch of {len(batch)} variants")
        batch.clear()


# Write remaining rows
if batch:
    file_exists = os.path.exists(output_path)

    with open(output_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=batch[0].keys())

        if not file_exists:
            writer.writeheader()

        writer.writerows(batch)

    print(f"Saved final batch of {len(batch)} variants")


    # file_exists = os.path.exists(OUTPUT_DIR / output_file)
    # with open(OUTPUT_DIR / output_file, "a", newline="", encoding="utf-8") as f:
    #     writer = csv.DictWriter(f, fieldnames=row.keys())
    #     if not file_exists:
    #         writer.writeheader()
    #     writer.writerow(row)

    # print(f"Saved variant {processed}/{len(variants)}: {variant} | cost={cost}")

Skipping 1/16471: ('A_Create Application', 'A_Submitted', 'W_Handle leads', 'A_Concept', 'W_Handle leads', 'W_Complete application', 'W_Complete application', 'W_Complete application', 'A_Accepted', 'O_Create Offer', 'O_Created', 'A_Complete', 'O_Sent (mail and online)', 'W_Complete application', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'A_Cancelled', 'O_Cancelled', 'W_Call after offers')
Skipping 2/16471: ('A_Create Application', 'A_Submitted', 'W_Handle leads', 'A_Concept', 'W_Handle leads', 'W_Complete application', 'W_Complete application', 'A_Accepted', 'O_Create Offer', 'O_Created', 'A_Complete', 'O_Sent (mail and online)', 'W_Complete application', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Validate application', 'W_Validate application', 'A_Validating', 'O_Returned', 'W_Validate application', 'A_Pending', 'O_Accepted', 'W_Validate application')
Skipping 

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x10b2ba7f0>>
Traceback (most recent call last):
  File "/Users/yeeyoo/Library/Python/3.9/lib/python/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


Processed 2395/16471: ('A_Create Application', 'A_Concept', 'W_Complete application', 'W_Complete application', 'A_Accepted', 'W_Complete application', 'W_Complete application', 'O_Create Offer', 'O_Created', 'A_Complete', 'O_Sent (mail and online)', 'W_Complete application', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'O_Create Offer', 'O_Created', 'O_Sent (online only)', 'W_Call after offers', 'W_Validate application', 'W_Validate application', 'A_Validating', 'O_Returned', 'W_Validate application', 'W_Validate application', 'W_Validate application', 'W_Validate application', 'W_Validate application', 'W_Validate application', 'A_Pending', 'O_Accepted', 'W_Validate application', 'O_Cancelled') | cost=3
Processed 2396/16471: ('A_Create Application', 'A_Submitted', 'W_Handle leads', 'A_Concept', 'W_Handle leads', 'W_Complete application', 'W_Complete application', 'W_Complete application', 'W_Complete application', 'W_Complete application', 'W_Complete applicat

KeyboardInterrupt: 

In [ ]:
import os
import csv
import json
import pandas as pd
from multiprocessing import cpu_count
from pathos.multiprocessing import ProcessingPool as Pool


# -----------------------------
# CONFIG
# -----------------------------
output_file = f"alignment_per_variant_{CASE}.csv"
output_path = OUTPUT_DIR / output_file

BATCH_SIZE = 200
N_WORKERS = 4
# N_WORKERS = 2


# -----------------------------
# LOAD CHECKPOINT
# -----------------------------
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    done_variants = set(df_existing["variant"].astype(str))
else:
    done_variants = set()


# -----------------------------
# WORKER FUNCTION
# -----------------------------
def process_variant(item):

    variant, traces = item

    representative = traces[0]

    alignment = aligner.apply_trace(
        representative,
        net_clean,
        im_clean,
        fm_clean
    )

    aligned = alignment["alignment"]

    cost, model_moves, log_moves = classify_and_cost(aligned)

    return {
        "variant": str(variant),
        "frequency": len(traces),
        "cost": cost,
        "alignment": json.dumps(aligned),
        "model_moves": json.dumps(model_moves),
        "log_moves": json.dumps(log_moves)
    }


# -----------------------------
# FILTER COMPLETED VARIANTS
# -----------------------------
items = [
    (v, t)
    for v, t in variants.items()
    if str(v) not in done_variants
]

print(f"Total variants: {len(variants)}")
print(f"Remaining after checkpoint: {len(items)}")


# -----------------------------
# EXECUTION
# -----------------------------
batch = []
processed = 0

file_exists = os.path.exists(output_path)

pool = Pool(nodes=N_WORKERS)

with open(output_path, "a", newline="", encoding="utf-8") as f:

    writer = None

    for result in pool.uimap(process_variant, items):

        processed += 1

        print(
            f"Processed {processed}/{len(items)}: "
            f"{result['variant']} | cost={result['cost']}"
        )

        batch.append(result)

        if writer is None:
            writer = csv.DictWriter(
                f,
                fieldnames=result.keys()
            )

            if not file_exists:
                writer.writeheader()
                file_exists = True

        if len(batch) >= BATCH_SIZE:

            writer.writerows(batch)
            f.flush()

            batch.clear()

    if batch:
        writer.writerows(batch)
        f.flush()

pool.close()
pool.join()
pool.clear()

print("Done.")

Total variants: 16471
Remaining after checkpoint: 14637
Processed 1/14637: ('A_Create Application', 'A_Submitted', 'W_Handle leads', 'A_Concept', 'W_Handle leads', 'W_Complete application', 'W_Complete application', 'W_Complete application', 'A_Accepted', 'O_Create Offer', 'O_Created', 'A_Complete', 'O_Sent (mail and online)', 'W_Complete application', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'O_Cancelled', 'A_Cancelled', 'W_Call after offers') | cost=1
Processed 2/14637: ('A_Create Application', 'A_Submitted', 'W_Handle leads', 'A_Concept', 'W_Handle leads', 'W_Complete application', 'W_Complete application', 'W_Complete application', 'A_Accepted', 'O_Create Offer', 'O_Created', 'A_Complete', 'O_Sent (mail and online)', 'W_Complete application', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Call after offers', 'W_Validate application', 'W_Va

### Task b+c: Aggregate per log and locate hotspots
- For each log, report two figures in a short table: the absolute fitness and the normalised fitness, where the worst-case alignment runs through the whole trace and then the shortest complete model run.
- For each log, count the model moves and log moves per activity across all cases, and report the top five activities by total in a short table: rows are hotspots, columns are activity, model moves and log moves.

In [ ]:
CSV_PATHS = {
    "clean":     "alignment_per_variant_clean.csv",
    "noised":    "alignment_per_variant_noised.csv",
    "recovered": "alignment_per_variant_recovered.csv",
}

In [ ]:
def shortest_model_run_cost(net, im, fm):
    """Align the EMPTY trace against the model. The resulting cost equals the
    length of the shortest complete firing sequence from im to fm. this is
    the 'worst case model run' needed for normalised fitness."""
    empty_log = EventLog([Trace()])
    alignment = aligner.apply_trace(empty_log[0], net, im, fm)
    aligned = alignment["alignment"]

    cost = 0
    for l, m in aligned:
        log_skip = (l == ">>")
        model_skip = (m == ">>")
        if log_skip and not model_skip:
            if m is not None:  # ignore invisible/tau transitions
                cost += 1
    return cost

In [ ]:
worst_case_model_cost = shortest_model_run_cost(net_clean, im_clean, fm_clean)
print(f"Shortest complete model run cost (model property): {worst_case_model_cost}")

Shortest complete model run cost (model property): 4


In [ ]:
def analyse_csv(log_name, csv_path, worst_case_model_cost, top_n):
    df = pd.read_csv(OUTPUT_DIR / csv_path)

    total_weighted_cost = 0
    total_weighted_worst_case = 0
    total_cases = 0

    model_move_counter = Counter()
    log_move_counter = Counter()

    for _, row in df.iterrows():
        freq = int(row["frequency"])
        cost = float(row["cost"])

        # trace length = number of activities in the variant tuple
        variant_tuple = ast.literal_eval(row["variant"])
        trace_len = len(variant_tuple)

        model_moves = json.loads(row["model_moves"])
        log_moves = json.loads(row["log_moves"])

        worst_case_i = trace_len + worst_case_model_cost # in the worst case, we have to do all the log moves (trace_len) and all the model moves (worst_case_model_cost)

        total_weighted_cost += freq * cost
        total_weighted_worst_case += freq * worst_case_i
        total_cases += freq

        for act in model_moves:
            model_move_counter[act] += freq
        for act in log_moves:
            log_move_counter[act] += freq

    # ---- Task 2b ----
    absolute_total_cost = total_weighted_cost
    normalised_fitness = 1 - (total_weighted_cost / total_weighted_worst_case)

    aggregate_row = {
        "log": log_name,
        "n_cases": total_cases,
        "n_variants": len(df),
        "absolute_total_cost": int(absolute_total_cost),
        "normalised_fitness": round(normalised_fitness, 4),
    }

    # ---- Task 2c ----
    all_activities = set(model_move_counter) | set(log_move_counter)
    hotspot_rows = []
    if len(all_activities) == 0:
        print(f"Warning: No activities found in log {log_name}. Skipping hotspot analysis.")
        df_hotspots = pd.DataFrame(columns=["activity", "model_moves", "log_moves"])
    else:
        for act in all_activities:
            mm = model_move_counter.get(act, 0)
            lm = log_move_counter.get(act, 0)
            hotspot_rows.append({"activity": act, "model_moves": mm, "log_moves": lm, "total": mm + lm})

        df_hotspots = (
            pd.DataFrame(hotspot_rows)
            .sort_values("total", ascending=False)
            .head(top_n)
            .drop(columns="total")
            .reset_index(drop=True)
        )

    return aggregate_row, df_hotspots

In [ ]:
aggregate_rows = []
hotspot_tables = {}

for log_name, csv_path in CSV_PATHS.items():
    agg_row, df_hotspots = analyse_csv(log_name, csv_path, worst_case_model_cost, top_n=5)
    aggregate_rows.append(agg_row)
    hotspot_tables[log_name] = df_hotspots

    print(f"=== Task 2c: Top-5 hotspot activities -- {log_name} ===")
    print(df_hotspots.to_string(index=False))

df_aggregate = pd.DataFrame(aggregate_rows)
df_aggregate.to_csv(OUTPUT_DIR / "2b_aggregate_fitness.csv", index=False)
print("=== Task 2b: Aggregate fitness per log ===")
print(df_aggregate.to_string(index=False))

for log_name, df_hotspots in hotspot_tables.items():
    df_hotspots.to_csv(OUTPUT_DIR / f"2c_hotspots_{log_name}.csv", index=False)

=== Task 2c: Top-5 hotspot activities -- clean ===
Empty DataFrame
Columns: [activity, model_moves, log_moves]
Index: []
=== Task 2c: Top-5 hotspot activities -- noised ===
              activity  model_moves  log_moves
W_Finalize application            0       1189
  W_Finish application            0       1166
            A_Complete            0        587
    W_Call after ofrs.            0        448
      W_Complete appl.            0        351
=== Task 2c: Top-5 hotspot activities -- recovered ===
  activity  model_moves  log_moves
A_Complete            0       2554
O_Accepted            0        351
 A_Pending            0        351
=== Task 2b: Aggregate fitness per log ===
      log  n_cases  n_variants  absolute_total_cost  normalised_fitness
    clean    31509       15930                    0              1.0000
   noised     2718        1279                 5753              0.9280
recovered     2554          18                 3256              0.9495
